In [2]:
import polars as pl
from procompa import get_project_root

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

## Input for homomultimer comparison pooled vs pair modles

get all proteins and their sequence

In [15]:
protein_AF_info = pl.read_parquet(data_dir/ "Homomultimer/Pipeline_prep/proteins.parquet") #mapping of pdb id to protein

In [16]:
# Assuming your dataframe is named d_f
with open(data_dir/ "iPTM_and_pLDDT/all_yeast_proteins_uniprot_mapped_sequences.csv", "w") as f:
    for uid, seq in protein_AF_info.select(["uniprot_id", "seq"]).iter_rows():
        f.write(f"{uid},{seq}\n")

After finding homomultimers

In [10]:
homomultimers = pl.read_csv(data_dir/ "Homomultimer/Pipeline_prep/homomultimers.csv") #mapping of pdb id to protein

In [11]:
#filter out matches,that potentially have Fusion tags, Affinity tags and cloning linker residues, or Partner proteins
homomultimers = homomultimers.filter((~pl.col("uniprot_id").str.contains(";") )& (pl.col("uniprot_seq_len") >= pl.col("seq_len")))

In [12]:
# keep match with best coverage
homomultimers = homomultimers.with_columns(
    coverage = (pl.col("uniprot_seq_len") / pl.col("seq_len"))
)

homomultimers = homomultimers.sort("coverage", descending=True).unique(subset=["uniprot_id"], keep="first")

In [13]:
homomultimers_CF = homomultimers.select([
    pl.format("HOMO_{}", pl.col("uniprot_id")).alias("#Complex ac"),
    pl.format(
        "{}({})", pl.col("uniprot_id"), pl.col("n_chains").cast(pl.Int64)
    ).alias("Identifiers (and stoichiometry) of molecules in complex"),
    pl.format("{} homomultimer", pl.col("uniprot_id")).alias(
        "Recommended name"
    ),
    pl.col("pdb_id"),
    pl.col("n_chains").cast(pl.Int64),
    pl.col("coverage").round(3),
])

In [ ]:
# 4. Write input for CF pipeline to TSV
homomultimers_CF.write_csv(data_dir/"Pipeline/6_sixth_subset_homomultimers_pool_vs_pair/sixth_input_homomultimers_pool_vs_pair.tsv", separator="\t")

Wrote 494 homomultimer complexes


Get sequences for all proteins that i have Homomultimer pdb files for 

In [ ]:
'''
find prot which are not :/cluster/project/beltrao/kdammer/master_thesis/data/iPTM_and_pLDDT/all_yeast_proteins_uniprot_mapped_sequences.csv
add them to csv (so get sequences)
create dataframe in correct input format:Complex ac = HOMO_<uniprot_id> (synthetic ID)
Identifiers (and stoichiometry) of molecules in complex = <uniprot_id>(0) (let Stoic predict freely) or <uniprot_id>(n_chains) 

'''


In [ ]:
homomultimers = 